# RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import shutil, os

# wipe the old L2-based store so it gets rebuilt with cosine
shutil.rmtree("../data/vector_store", ignore_errors=True)

In [2]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_1479/582141364.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Read all the PDF inside the directory

def process_all_pdfs(pdf_directory):
    """ Process all the PDF document in a directory """
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")


    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information into metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(all_documents)}")
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents


all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

 Processing: ../data/pdf/resume.pdf
Loaded 3

 Processing: ../data/pdf/assignment.pdf
Loaded 5

 Total documents loaded: 5


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-06-29T12:35:04-07:00', 'author': 'Vaishali SR', 'moddate': '2026-06-29T12:35:04-07:00', 'source': '../data/pdf/resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'resume.pdf', 'file_type': 'pdf'}, page_content='Vaishali Somu Ramesh  \n                                                      284 Weston Road, Toronto, Ontario, M6N 3P5 \n (226) 932-1241 \nvaishalisomu12feb@gmail.com  \nLinkedIn: https://www.linkedin.com/in/vaishali-somu-ramesh-6a739a107/   \n \nPROFESSIONAL SUMMARY  \nExperienced Fullstack Developer with a strong background in web development and Data Science. Proven expertise \nin developing scalable software solutions, performing data analysis, and implementing machine learning models. \nSkilled in various programming languages, frameworks, and cloud platforms. Passionate about continuous learning \nand applying new technologies to solve real-world problems.

# Chunking

In [ ]:
#### Text Splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG Performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\n Example Chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 5 documents into 17 chunks

 Example Chunk: 
Content: Vaishali Somu Ramesh  
                                                      284 Weston Road, Toronto, Ontario, M6N 3P5 
 (226) 932-1241 
vaishalisomu12feb@gmail.com  
LinkedIn: https://www.linkedin.c
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-06-29T12:35:04-07:00', 'author': 'Vaishali SR', 'moddate': '2026-06-29T12:35:04-07:00', 'source': '../data/pdf/resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-06-29T12:35:04-07:00', 'author': 'Vaishali SR', 'moddate': '2026-06-29T12:35:04-07:00', 'source': '../data/pdf/resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'resume.pdf', 'file_type': 'pdf'}, page_content='Vaishali Somu Ramesh  \n                                                      284 Weston Road, Toronto, Ontario, M6N 3P5 \n (226) 932-1241 \nvaishalisomu12feb@gmail.com  \nLinkedIn: https://www.linkedin.com/in/vaishali-somu-ramesh-6a739a107/   \n \nPROFESSIONAL SUMMARY  \nExperienced Fullstack Developer with a strong background in web development and Data Science. Proven expertise \nin developing scalable software solutions, performing data analysis, and implementing machine learning models. \nSkilled in various programming languages, frameworks, and cloud platforms. Passionate about continuous learning \nand applying new technologies to solve real-world problems.

# Embedding and Vector StoreDB

## Embeddings

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles documen embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize the embedding manager

            Args:
                model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embedding for a list of texts

            Args:
                texts: List of text strings o embed

            Returns:
                numpy array of embeddings wwith shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")  
        return embeddings
    
### Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1556.14it/s]


Model loaded successfully. Embedding dimension: 384


## Vector Store DB

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """ 
            Initialize the vector store

            Args:
                collection_name: Name of the ChromaDB collection
                persist_directory: Directory to persists the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and their embeddings to the vector store

            Args:
                documents: List of Langchain documents
                embeddings: COrresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match  number of the embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data do ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            metadatas.append(metadata)

            # Document Content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding document to vector store: {e}")
            raise

# Initialize vectorStore
vectorstore = VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-06-29T12:35:04-07:00', 'author': 'Vaishali SR', 'moddate': '2026-06-29T12:35:04-07:00', 'source': '../data/pdf/resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'resume.pdf', 'file_type': 'pdf'}, page_content='Vaishali Somu Ramesh  \n                                                      284 Weston Road, Toronto, Ontario, M6N 3P5 \n (226) 932-1241 \nvaishalisomu12feb@gmail.com  \nLinkedIn: https://www.linkedin.com/in/vaishali-somu-ramesh-6a739a107/   \n \nPROFESSIONAL SUMMARY  \nExperienced Fullstack Developer with a strong background in web development and Data Science. Proven expertise \nin developing scalable software solutions, performing data analysis, and implementing machine learning models. \nSkilled in various programming languages, frameworks, and cloud platforms. Passionate about continuous learning \nand applying new technologies to solve real-world problems.

### Convert Text to Embeddings

In [11]:
### Conver the text to embeddings
texts = [doc.page_content for doc in chunks]


### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts=texts)

### Store in the vector database
vectorstore.add_documents(documents=chunks, embeddings=embeddings)

Generating embeddings for 17 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.31it/s]

Generated embeddings with shape: (17, 384)
Adding 17 documents to vector store...
Sucessfully added 17 documents to vector store
Total documents in collection: 17


# RAG Retriever Pipeline from Vector Store




In [12]:
class RAGRetriever:
    """Handles query-based retrievak from vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
            Initialize the retriever

            Args:
                - vector_Store: vector sore containing document embeddings
                - embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
       
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query

            Args:
                - query: The search query
                - top_k: Number of top results to return
                score_threshold: Minimum similarity score_threshold

            Returns:
                List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(texts=[query])[0]

        # Search in vector_store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process Results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        
        except Exception as e:
            print(f"Something is not right! {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)


In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("PDF content")

Retrieving documents for query: 'PDF content'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.37it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2f7f69c7_14',
  'content': 'You are working on an AI-powered document assistant tasked with retrieving meaningful \ninsights from long documents. To build this assistant, you must: \n• Load and chunk multiple PDF files \n• Convert the chunks into embeddings using OpenAI models \n• Store the embeddings in an FAISS vector database \n• Query the vectorstore to retrieve the most relevant chunk for a given user question \n \nTask \n \nYou will build the entire RAG pipeline from scratch in Visual Studio Code, following a step-by-\nstep process: \n \n1. Chunk and embed PDF content: Load and chunk your PDF documents, and generate \nembeddings using OpenAI’s embedding model \n2. Store embeddings in FAISS: Store the document chunks in an FAISS vectorstore to \nenable similarity-based retrieval \n3. Run retrieval queries: Accept a sample query and retrieve the most relevant chunk \nusing semantic search',
  'metadata': {'total_pages': 2,
   'creator': 'Microsoft® Word for Microsoft 3

# Integration VectorDB context pipeline with LLM Output

In [ ]:
### Simple RAG pipeline with Groq LLM ---
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever:RAGRetriever, llm, top_k = 3):
    # Retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevan context found to answer the question"
    
    # generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
                Context: {context}
                Question: {query}
                Answer:
            """
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [16]:
user_prompt = input("Enter the value: ")
answer = rag_simple(user_prompt, rag_retriever, llm)
print(answer)

Retrieving documents for query: ''
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Based on the provided context, here are concise answers to potential questions related to software scalability and security:

1. **What programming languages and frameworks have you used for scalable backend services?**
   - Ruby on Rails, Python (with Pytest and unittest), and JavaScript (with ReactJS and MeteorJS).

2. **How do you ensure high code coverage and application reliability?**
   - Developed and maintained automated tests using Pytest and unittest, ensuring high code coverage.

3. **What experience do you have with asynchronous request handling and message queues?**
   - Integrated message queues such as Kafka and RabbitMQ for reliable asynchronous, event-driven communication between services.

4. **How do you manage CI/CD pipelines and apply DevOps practices?**
   - Managed CI/CD pipelines and applied DevOps practices to streamline builds, testing, and deployments.

5. **What experience do you have with code reviews and mentoring junior engineers?**
   - Participated in c

# Enhanced RAG Pipeline Features

In [17]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query: str, retriever: RAGRetriever, llm: ChatGroq, top_k=5, min_score=0.2, return_context=False):
    """
        RAG pipeline with extra feature:
            - Returns answer, sources, confidence_score, and optionally full context
    """
    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant context found.", "source": [], "confidence": 0.0, "context": ""
        }
    
    # Prepare context and sources
    context = "\n\n".join([doc["content"] for doc in results])
    sources = [
        {
            "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:120] + "..."
        } for doc in results
    ]
    confidence = max([doc["similarity_score"] for doc in results])

    # Generate answer
    prompt = f"""
        Use the following context to answer the question concisely. 
        Context: {context}
        Question: {query}
        Answer:
    """
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }
    if return_context:
        output["context"] = context
    return output

# Example Usage
user_question = input("Enter your question for Advanced RAG Pipeline: ")
result = rag_advanced("Who is Vaishali?", rag_retriever, llm, top_k=3, min_score=0.15, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence: {result['confidence']}")
print(f"Context Preview: {result['context'][:300]}")

Retrieving documents for query: 'Who is Vaishali?'
Top K: 3, Score Threshold: 0.15
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: Vaishali Somu Ramesh is a Fullstack Developer with expertise in web development, Data Science, and machine learning.
Sources: [{'source': 'resume.pdf', 'page': 0, 'score': 0.2471790909767151, 'preview': 'Vaishali Somu Ramesh  \n                                                      284 Weston Road, Toronto, Ontario, M6N 3P5 ...'}]
Confidence: 0.2471790909767151
Context Preview: Vaishali Somu Ramesh  
                                                      284 Weston Road, Toronto, Ontario, M6N 3P5 
 (226) 932-1241 
vaishalisomu12feb@gmail.com  
LinkedIn: https://www.linkedin.com/in/vaishali-somu-ramesh-6a739a107/   
 
PROFESSIONAL SUMMARY  
Experienced Fullstack Developer wi


# Advanced RAG Pipeline: Streaming, Citations, History, Summarization

In [18]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever: RAGRetriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = [] # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve Relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not result:
            answer = "Norelevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc["content"] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""
                Use the following context to answer the question concisely.
                    Context: {context} 
                    Question: {question}
                    Answer:
                """
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.60it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:

                Use the following context to answer the question concisely.
                    Context: Senior Fullstack Developer 
• Led development of scalable backend services using Ruby on Rails and frontend using VueJs, Typescript, 
building RESTful APIs for high-volume transaction systems for 200+ global brands

.  
• Built and maintained Rails applications following MVC architecture and REST principles. 
• Architected and optimized complex database queries using ActiveRecord and PostgreSQL and MySQL and 
ActiveRecord patterns reducing N+1 query issues and improving API response times by 60%. 
• Created and maintained Rails migrations to support evolving database schemas. 
• Engineered asynchronous background processing using Sidekiq and Redis, ensuring reliable execution of 
mission-critical tasks and system stability. 
• Collaborated cross-functionally with Product Managers and Designers to translate roadmap visions into 
technical specifications and production-ready code. 
• Mentored junior engineers through rigorous code reviews and architectural planning sessions, fostering a

debugging skills. 
• Built internal Rails tools for production support and operational workflows. 
• Wrote unit and integration tests using RSpec to ensure application reliability. 
• Implemented caching strategies 